# TTS content-error verifier: end-to-end run

Reproduces every result in the [README](../README.md) on a Colab T4 GPU.

**Question.** Do ASR-based checks catch text-to-speech content errors (omitted, repeated, substituted or cut-off words), and does Whisper's language model hide errors that a CTC recogniser without a language model still reports?

**Ground truth.**

| Source | Clips | Role |
|---|---|---|
| Synthetic corruptions | 320 | Text is corrupted before synthesis; the original prompt is the reference. Measures detector sensitivity with known answers. |
| Blind human labels | 40 | Natural clips labelled by ear, stratified by category × model. Validates the automated judge. |
| Gemini audio judge | 200 | Labels all natural clips. Automated; reported separately from human labels. |

Outputs and labels are stored on Google Drive, and every step is resumable: re-running a cell skips work that is already done.

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


## 1. Setup
Unpacks the repository from a zip on Google Drive and links `outputs/` and `labels/` to Drive so they persist across sessions.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/tts-content-verifier'
!rm -rf /content/repo && unzip -q /content/drive/MyDrive/tts-content-verifier.zip -d /content && mv /content/tts-content-verifier /content/repo
%cd /content/repo
!mkdir -p {DRIVE}/outputs {DRIVE}/labels
!cp -n labels/* {DRIVE}/labels/ 2>/dev/null; rm -rf outputs labels
!ln -s {DRIVE}/outputs outputs && ln -s {DRIVE}/labels labels
!ls scripts && ls outputs | head

Mounted at /content/drive
/content/repo
01_make_prompts.py  04_build_labeler.py  07_synthetic_errors.py
02_generate.py	    05_evaluate.py	 08_gemini_judge.py
03_transcribe.py    06_regenerate.py	 09_retranscribe_ctc.py
clips
env.txt
generations.jsonl
labeler
labeler.zip
synthetic
synthetic_predictions.json
synthetic_results.md
transcripts.jsonl


## 2. Install
`transformers` is pinned to a range compatible with Coqui XTTS. Package versions are recorded in `outputs/env.txt`.

In [3]:
!pip install -q -r requirements-gpu.txt
!pip install -q "transformers>=4.47,<5" google-genai
%env COQUI_TOS_AGREED=1
!pip freeze > outputs/env.txt
!python -m unittest discover tests

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.6/107.6 kB 8.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 90.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 862.8/862.8 kB 58.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 82.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 36.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.2/56.2 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.8/168.8 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 997.3/997.3 kB 68.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 639.3/639.3 kB 47.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.5/163.5 kB 17.6 MB/s eta 0:00:

## 3. Natural clips
100 prompts × 2 TTS models (XTTS-v2, F5-TTS) = 200 clips, each transcribed by Whisper large-v3-turbo and greedy-CTC wav2vec2.

In [4]:
!python scripts/01_make_prompts.py
!python scripts/02_generate.py
!python scripts/03_transcribe.py
!wc -l outputs/generations.jsonl outputs/transcripts.jsonl

Wrote 100 prompts to /content/repo/data/prompts.jsonl
  long               dev  10
  long               test 10
  numbers_names      dev  20
  numbers_names      test 12
  ordinary           dev  16
  ordinary           test 12
  repetition_prone   dev  10
  repetition_prone   test 10
I0000 00:00:1792404916.936086    1994 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1792404919.427900    1994 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
100% 1.87G/1.87G [00:21<00:00, 86.6MiB/s]
4.37kiB [00:00, 10.1MiB/s]
361kiB [00:00, 63.6MiB/s]
100% 32.0/32.0 [00:00<00:00, 188kiB/s]
100% 7.75M/7.75M [00:00<00:00, 20.5MiB/s]
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = r

## 4. Synthetic benchmark: generation
40 prompts, each synthesised clean and with one deliberate omission, repetition and substitution, by both models (320 clips). The original prompt is the reference, so the ground truth is known by construction.

In [5]:
!python scripts/07_synthetic_errors.py --generate --n-prompts 40

I0000 00:00:1792405002.922765    2499 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1792405005.375679    2499 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
Download Vocos from huggingface charactr/vocos-mel-24khz

vocab :  /usr/local/lib/python3.13/dist-packages/f5_tts/infer/examples/vocab.txt
token :  custom
model :  /root/.cache/huggingface/hub/models--SWivid--F5-TTS/snapshots/84e5a410d9cead4de2f847e7c9369a6440bdfaca/F5TTS_v1_Base/model_1250000.safetensors 



## 5. CTC re-transcription and synthetic evaluation
wav2vec2 garbles the first word of clips that start abruptly ("htrain left…" for "The train left…"). CTC is re-run on every clip with 0.5 s of leading silence; the original output is kept as `ctc_text_nopad` for comparison. Alignment also tolerates spelling variants (grey/gray) and split compounds (wrist watches).

Output: `outputs/synthetic_results.md`. The clean rows are false-alarm rates.

In [6]:
!python scripts/09_retranscribe_ctc.py
!python scripts/07_synthetic_errors.py --evaluate

I0000 00:00:1792405049.105578    2785 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1792405051.563240    2785 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
preprocessor_config.json: 100% 158/158 [00:00<00:00, 1.35MB/s]
tokenizer_config.json: 100% 162/162 [00:00<00:00, 1.56MB/s]
config.json: 1.61kB [00:00, 8.47MB/s]
vocab.json: 100% 291/291 [00:00<00:00, 3.14MB/s]
special_tokens_map.json: 100% 85.0/85.0 [00:00<00:00, 755kB/s]
pytorch_model.bin: 100% 1.26G/1.26G [00:08<00:00, 149MB/s]
Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-large-960h-lv60-self and are newly initialized: ['wav2vec2.masked_spec_embed']
You should probably TRAIN this model on a down-stream task to b

## 6. Blind human labels
A fixed-seed sample of 40 natural clips, 5 per (prompt category × TTS model). Only the prompt and audio are shown: no model name, transcript or judge output. Labelling rules: `labels/LABELING.md`.
- **Error:** a prompt word is missing, repeated, replaced or cut off, or an extra word is spoken.
- **Not an error:** odd stress or accent on a recognisable word; a valid alternative reading of a number, time or code.

The first cell displays the clips. The second records one answer per clip in order: `1` clean, `2` error, with an optional type (`2o` omission, `2r` repetition, `2s` substitution, `2t` truncation, `2i` insertion). Labels are written to `labels/primary.csv`.

In [10]:
import sys, csv; sys.path.insert(0, 'src')
from IPython.display import Audio, Markdown, display
from tts_verifier.listen import validation_sample
from tts_verifier.dataset import clip_key
from tts_verifier.io import LABELS
S = validation_sample(40)
for i, g in enumerate(S, 1):
    display(Markdown(f"**{i}.** {g['text']}"))
    display(Audio(f"outputs/{g['wav']}"))

In [11]:
ANSWERS = """
1 1 1 1 2i
1 1 1 1 1
1 1 1 1 2s
1 1 1 1 1
1 1 1 2o 1
2o 1 1 1 1
1 1 1 1 1
1 1 1 1 1
"""
TYPES = {'o':'omission','r':'repetition','s':'substitution','t':'truncation','i':'insertion'}
toks = ANSWERS.split()
assert len(toks) == 40 and all(t[0] in '12' for t in toks)
with open(LABELS/'primary.csv', 'w', newline='') as f:
    w = csv.writer(f); w.writerow(['clip_key','error','types','notes'])
    for g, t in zip(S, toks):
        w.writerow([clip_key(g['clip_id']), int(t[0]=='2'), ';'.join(TYPES[c] for c in t[1:] if c in TYPES), ''])
print(f"Saved {len(toks)} labels; errors: {sum(t[0]=='2' for t in toks)}")

Saved 40 labels; errors: 4


## 7. Gemini audio judge
Gemini receives each natural clip with its target text, transcribes it verbatim, and labels it clean or error (`labels/gemini.csv`). It is then compared with the human labels (`outputs/gemini_agreement.md`), and the detector evaluation is run with its labels (`outputs/results_gemini.md`).

Requires a Gemini API key, entered at the prompt. The first run stopped at the free-tier limit of 20 requests per model per day; the second completed the remaining clips on a paid tier (a few pence) and resumed from the clips already judged.

In [12]:
import os
from getpass import getpass
if not os.environ.get("GEMINI_API_KEY"):
    os.environ["GEMINI_API_KEY"] = getpass("Gemini API key: ")
GEMINI_MODEL = "gemini-3.8-flash"

!python scripts/08_gemini_judge.py --judge --model {GEMINI_MODEL}
!python scripts/08_gemini_judge.py --agree
!python scripts/05_evaluate.py --labels gemini.csv --tag _gemini

Gemini API key: ··········
024b2b95ad 1
872a2d6df1 1
7484d17e62 1
541a391c1f 0
5aafa3e056 0
e5c4957b0b 0
15abbf2dea 1
0809b53a68 0
f183c17c43 0
fd42b52e41 0
30e89e4100 0
156310aed3 1
c97a8c1fae 1
b8b00f343e 1
547256ae5d 0
d128e0476f 1
f525ec18e9 0
738d28bace 0
dc78dfe417 0
75f0263abc 0
372121eb72 0
retry 20487ac410 ClientError("429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check y
retry 20487ac410 ClientError("429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check y
retry 20487ac410 ClientError("429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check y
retry e0383255b4 ClientError("429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check y
retry e0383255b4 ClientError("429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check y
retry e038

In [14]:
!wc -l labels/primary.csv
!python scripts/08_gemini_judge.py --judge --model gemini-3.8-flash --gap 1
!python scripts/08_gemini_judge.py --agree
!python scripts/05_evaluate.py --labels gemini.csv --tag _gemini

41 labels/primary.csv
20487ac410 0
e0383255b4 0
40daa1b688 0
4ba6b04998 0
829e9dd968 0
3e63e72640 0
24be9d77aa 0
e2b4db9408 0
ba3ae61db1 0
283a4eafaa 0
629597c483 0
bde60bb86b 0
385418e478 0
c39172504a 0
7f14fc56c8 0
163b3de8b8 0
6ef7f064a8 0
ad388693a5 0
999e6e1d8e 0
bb5b667eca 0
d208a61b8d 0
c5011ffa27 0
6aac40655d 0
606715cb17 0
b66ef1ce1f 0
bdede4d90f 0
b7fa56053c 1
ff50504f94 1
04cd261fc0 1
4cff810c90 1
f2c7f0e90f 0
28bd8562dd 0
b1f8f499bd 0
328950e2c2 0
6d582e7b67 1
2697b4dc06 1
fe0bce6357 0
d110231d94 1
66be2aab1e 0
555438e98e 1
5dcbcbd4ca 0
27bb298d7d 1
b08f9c656c 0
c6683505ba 0
4b284d3e27 0
a16843a0ea 0
9c8fca51f6 0
c3c255be68 0
1a4d25b256 0
75c61f7080 0
3a16dde0c8 0
0caaa1b2c3 0
a02228aa42 0
96eaa514b8 0
79b9900a48 0
cf75e24ea9 0
83a86ea7a8 0
b7fe8fce49 0
f55535807e 0
09357fedcd 0
6e98f38b07 0
a875a9856b 0
459d0526fc 0
05bd99a5c8 0
91028a624c 0
5114821d69 0
664ea09fa6 0
071c1ed257 1
ee50f7282e 1
37a219bed7 1
273a3bc1d6 0
e2af2b6988 1
59fc7dbccd 0
9e516b1871 1
5339f4c151 1
23c

## 8. Results
Prints the result files and bundles them with the labels and the environment record.

In [15]:
import os
for f in ["synthetic_results.md", "gemini_agreement.md", "results_gemini.md"]:
    print(f"\n{'=' * 30} {f} {'=' * 30}\n")
    print(open(f"outputs/{f}").read() if os.path.exists(f"outputs/{f}") else "(not produced)")

!zip -qj outputs/results_bundle.zip outputs/synthetic_results.md outputs/gemini_agreement.md outputs/results_gemini.md outputs/results_gemini.json outputs/synthetic_predictions.json outputs/env.txt labels/primary.csv labels/gemini.csv
from google.colab import files
files.download("outputs/results_bundle.zip")


============================== synthetic_results.md ==============================

# Synthetic benchmark (NOT human-verified)

Text was deliberately corrupted before synthesis; the original prompt is the reference. Ground truth is the intended corruption. This tests sensitivity to deliberate content changes, not performance on naturally occurring TTS errors. The TTS model may add its own errors.

Clips: 320; corrupted: 240; clean controls: 80

## Detection rate by corruption type (95% CI over prompt groups)

| TTS model | Corruption | n | whisper | ctc | either |
|---|---|---|---|---|---|
| f5_tts | clean (false-alarm rate) | 40 | 18% [5, 35] | 18% [5, 35] | 22% [7, 42] |
| f5_tts | omission (recall) | 40 | 100% [100, 100] | 100% [100, 100] | 100% [100, 100] |
| f5_tts | repetition (recall) | 40 | 88% [77, 96] | 85% [71, 96] | 90% [81, 98] |
| f5_tts | substitution (recall) | 40 | 100% [100, 100] | 100% [100, 100] | 100% [100, 100] |
| xtts_v2 | clean (false-alarm rate) | 40 | 20% [5

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>